# Return timing and extreme returns

In [1]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from eda_data import *

In [2]:
lf = chars_lazy().select("permno", "DATE", "ym", "mom1m", "mvel1", "mom12m")
r = returns_lazy()
def shift_ym(k):
    d = pl.date(pl.col("ym") // 100, pl.col("ym") % 100, 1).dt.offset_by(f"{k}mo")
    return (d.dt.year() * 100 + d.dt.month()).alias("ym")
panel = lf.with_columns(pl.col("mvel1").shift(1).over("permno", order_by="DATE").alias("mvel1_prev"),
                        pl.col("DATE").shift(1).over("permno", order_by="DATE").alias("DATE_prev"))
for k, name in [(-1, "ret_tm1"), (0, "ret_t"), (1, "ret_tp1")]:
    panel = panel.join(r.with_columns(shift_ym(-k)).rename({"ret": name}), on=["permno", "ym"], how="left")
panel = panel.filter(pl.col("DATE") <= IS_END_DATE).collect()
panel.height

1463156

## Correlation of mom1m (row dated t) with returns dated t-1, t, t+1

In [3]:
def corr_pooled(x, y, d):
    d = d.filter(pl.col(x).is_not_null() & pl.col(y).is_not_null() & pl.col(y).is_not_nan())
    return pl.DataFrame({
        "pearson": [d.select(pl.corr(x, y)).item()],
        "pearson_ret_within_1pct_99pct": [d.filter((pl.col(y) > pl.col(y).quantile(0.01)) & (pl.col(y) < pl.col(y).quantile(0.99))).select(pl.corr(x, y)).item()],
        "spearman": [d.select(pl.corr(x, y, method="spearman")).item()], "n": [d.height]})
pl.concat([corr_pooled("mom1m", y, panel).with_columns(pl.lit(y).alias("return")) for y in ["ret_tm1", "ret_t", "ret_tp1"]])

pearson,pearson_ret_within_1pct_99pct,spearman,n,return
f64,f64,f64,i64,str
0.9585,0.9964,0.9976,664899,"""ret_tm1"""
-0.0057,-0.0140,0.0014,665165,"""ret_t"""
-0.0106,-0.0052,0.0016,662332,"""ret_tp1"""


## Correlation of log(mvel1_t / mvel1_{t-1}) with returns dated t-1, t, t+1 (consecutive months only)

In [4]:
p2 = panel.filter((pl.col("DATE") // 100 - pl.col("DATE_prev") // 100).is_in([1, 89])).with_columns(
    (pl.col("mvel1") / pl.col("mvel1_prev")).log().alias("dlog_mvel1"))
pl.concat([corr_pooled("dlog_mvel1", y, p2).with_columns(pl.lit(y).alias("return")) for y in ["ret_tm1", "ret_t", "ret_tp1"]])

pearson,pearson_ret_within_1pct_99pct,spearman,n,return
f64,f64,f64,i64,str
NaN,NaN,0.9581,664540,"""ret_tm1"""
NaN,NaN,0.0010,662052,"""ret_t"""
NaN,NaN,-0.0002,659234,"""ret_tp1"""


## Monthly rank IC of mom1m and mom12m vs each return date

In [5]:
rows = []
for x in ["mom1m", "mom12m"]:
    for y in ["ret_tm1", "ret_t", "ret_tp1"]:
        m = (panel.filter(pl.col(x).is_not_null() & pl.col(y).is_not_null() & pl.col(y).is_not_nan())
                  .group_by("DATE").agg(pl.corr(x, y, method="spearman").alias("ic"))["ic"])
        rows.append({"char": x, "return": y, "ic_mean": m.mean(), "ic_t": m.mean() / (m.std() / np.sqrt(m.len()))})
pl.DataFrame(rows)

char,return,ic_mean,ic_t
str,str,f64,f64
"""mom1m""","""ret_tm1""",0.9973,7478.6499
"""mom1m""","""ret_t""",-0.0256,-3.6179
"""mom1m""","""ret_tp1""",0.0028,0.4127
"""mom12m""","""ret_tm1""",0.0386,4.2173
"""mom12m""","""ret_t""",0.0547,6.2134
"""mom12m""","""ret_tp1""",0.0505,6.1724


## Monthly return distribution (full returns file)

In [6]:
ret = pl.scan_csv(RAW_RETURNS_CSV).select("permno", "DATE", pl.col("monthly_ret").cast(pl.Float64, strict=False).alias("ret")).collect()
pl.DataFrame({
    "rows": [ret.height], "null_or_unparsable": [ret["ret"].null_count()], "nan": [ret["ret"].is_nan().sum()],
    "below_-0.9": [(ret["ret"] < -0.9).sum()], "above_1": [(ret["ret"] > 1).sum()], "above_5": [(ret["ret"] > 5).sum()],
    "min": [ret["ret"].min()], "max": [ret["ret"].max()]})

rows,null_or_unparsable,nan,below_-0.9,above_1,above_5,min,max
i64,i64,i64,i64,i64,i64,f64,f64
1272137,0,0,14,1487,12,-0.9813,13.4951


In [7]:
q = [0.0001, 0.001, 0.01, 0.05, 0.5, 0.95, 0.99, 0.999, 0.9999]
pl.DataFrame({"quantile": q, "ret": [ret["ret"].drop_nans().quantile(x) for x in q]})

quantile,ret
f64,f64
0.0001,-0.7778
0.0010,-0.5774
0.0100,-0.3330
0.0500,-0.1757
0.5000,0.0067
0.9500,0.2010
0.9900,0.4258
0.9990,1.0556
0.9999,2.4000


In [8]:
ret.drop_nans().sort("ret", descending=True).head(15)

permno,DATE,ret
i64,i64,f64
91186,20090529,13.4951
89169,20170630,8.3365
84384,19980930,8.0000
68734,19910228,6.5000
45306,19991231,6.4074
79261,19971031,6.0769
61372,19920131,5.7692
91427,20080731,5.5517
90816,20090130,5.2500
